# AF2 complementary mechanisms — static audit
Audit AF2CTRL, AF2FS1, AF2SFS1, dan AF2BHCL1. Tidak melakukan training atau membuka test.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import importlib,os,shutil,subprocess,sys,torch
from pathlib import Path
assert torch.cuda.is_available(),'Aktifkan T4 GPU.'
REPO=Path('/content/coffee-bean-detection'); BRANCH='codex/af2-complementary-mechanisms'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
clone=['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(3):
    result=subprocess.run(clone)
    if result.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
else: raise RuntimeError('Git clone gagal tiga kali.')
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
from coffee_detector.drive_project import resolve_drive_project_root,require_project_artifact
AF2_REL='experiments/faruq-v3-breadth-screening-batch-v1/candidates/AFAB/AF2_seed42/weights/best.pt'
PROJECT=resolve_drive_project_root(required_relative_paths=(AF2_REL,))
AF2=require_project_artifact(PROJECT,AF2_REL)
OUTPUT=PROJECT/'experiments/faruq-v3-af2-complement-v1'; STATIC=OUTPUT/'static_audit.json'
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2_complement_audit','--af2-checkpoint',str(AF2),'--output',str(STATIC),'--device','0']
print('MENJALANKAN STATIC AUDIT:', ' '.join(command),flush=True)
subprocess.run(command,cwd=REPO,check=True)
print('STATIC:',STATIC)
